# task_10 Solution

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

base = Path("../data")


In [ ]:

depots = pd.read_csv(base / "depots.csv")
scans = pd.read_csv(base / "scans.csv")
shipments = pd.read_csv(base / "shipments.csv")

shipments["ship_date"] = pd.to_datetime(shipments["ship_date"])
scans["scan_time"] = pd.to_datetime(scans["scan_time"])

first_events = scans.sort_values("scan_time").groupby(["shipment_id", "status"])["scan_time"].first().unstack()
ship = shipments.merge(first_events, on="shipment_id")
ship["delay_days"] = (ship["delivered"].dt.floor("D") - ship["ship_date"]).dt.days - ship["promised_days"]
ship["on_time"] = ship["delay_days"] <= 0

q1 = str(ship[ship["weight_kg"] > 20].groupby("carrier")["on_time"].mean().sort_values(ascending=False).index[0])

south_depots = set(depots[depots["region"] == "South"]["depot_id"])
south_ship = ship[ship["origin_depot"].isin(south_depots)].copy()
south_ship["pickup_to_hub_hours"] = (south_ship["hub_received"] - south_ship["picked_up"]).dt.total_seconds() / 3600
q2 = str(south_ship["pickup_to_hub_hours"].median())

q3 = str(ship.groupby("dest_depot")["delay_days"].mean().sort_values(ascending=False).index[0])

in_transit_counts = scans[scans["status"] == "in_transit"].groupby("shipment_id").size()
exactly_two = set(in_transit_counts[in_transit_counts == 2].index)
subset = ship[ship["shipment_id"].isin(exactly_two)]
q4 = f"{(100 * (subset['delay_days'] > 0).mean()):.1f}%"

q5 = ship[ship["on_time"]].groupby("ship_date")["weight_kg"].sum().sort_values(ascending=False).index[0].date().isoformat()


Q1: Among shipments with weight_kg > 20, which carrier has the highest on-time delivery rate, where on-time means floor(delivered_time to calendar date) - ship_date <= promised_days?

In [ ]:
print(q1)


Q2: For shipments originating from depots in the South region, what is the median number of hours between picked_up and hub_received?

In [ ]:
print(q2)


Q3: Which dest_depot has the largest average delay_days, where delay_days = (floor(delivered_time to calendar date) - ship_date) - promised_days?

In [ ]:
print(q3)


Q4: What percentage of shipments with exactly 2 in_transit scans were delivered late, rounded to 1 decimal place?

In [ ]:
print(q4)


Q5: Among on-time shipments, which ship_date has the highest total weight_kg shipped?

In [ ]:
print(q5)
